#### What we will use
1. IAM roles and users
2. S3 buckets
3. Complete Infrastracture of AWS Sagemaker - Training, Endpoints


In [ ]:
import os

os.environ["OS_OPT"] = "linux"

import sagemaker
from sagemaker.core.helper.session_helper import Session
from sklearn.model_selection import train_test_split
import boto3
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
role = os.getenv("SAGEMAKER_ROLE_ARN")

sm_boto3 = boto3.client("sagemaker")
sess = Session()
region = sess.boto_session.region_name
bucket = "mobbucketsagemaker123321"
print("Using bucket" + bucket)

In [ ]:
print(region)

In [ ]:
df = pd.read_csv("mob_price_classification_train.csv")
df.head()

df.shape

In [ ]:
df.isnull().sum()

In [ ]:
df['price_range'].value_counts()

In [ ]:
df.columns

In [ ]:
features = list(df.columns)
features

In [ ]:
label = features.pop(-1)
label

In [ ]:
x=df[features]
y=df[label]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(x,y,test_size=0.15,random_state=0)


In [ ]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
trainX = pd.DataFrame(X_train)
trainX[label] = y_train

testX=pd.DataFrame(X_test)
testX[label] = y_test

In [ ]:
trainX

In [ ]:
trainX.to_csv("train-V-1.csv", index=False)
testX.to_csv("test-V-1.csv", index=False)

In [ ]:
bucket

In [ ]:
## send data to S3. Sagemkaer will take the data for training from s3
sk_prefix="sagemaker/mobile_price_classification/sklearncontainer"
trainpath=sess.upload_data(path='train-V-1.csv', bucket=bucket, key_prefix = sk_prefix)
testpath=sess.upload_data(path='test-V-1.csv', bucket=bucket, key_prefix = sk_prefix)
print(trainpath)
print(testpath)

## Script used by AWS Sagemaker to train models

In [ ]:
%%writefile src/script.py

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score
import sklearn
import joblib
import boto3
import pathlib
from io import StringIO
import argparse
import os
import numpy as np
import pandas as pd

def model_fn(model_dir):
    clf = joblib.load(os.path.join(model_dir, "model.joblib"))
    return clf


if __name__=="__main__":
    print("[Info] Extracting arguments")
    parser = argparse.ArgumentParser()

    # Hyperparameter
    parser.add_argument("--n-estimators", type=int, default=100)
    parser.add_argument("--random-state", type=int, default=0)

    ### Data, model and output directories
    parser.add_argument("--model-dir", type=str, default=os.environ.get("SM_MODEL_DIR"))
    parser.add_argument("--train", type=str, default=os.environ.get("SM_CHANNEL_TRAIN"))
    parser.add_argument("--test", type=str, default=os.environ.get("SM_CHANNEL_TEST"))
    parser.add_argument("--train-file", type=str, default="train-V-1.csv")
    parser.add_argument("--test-file", type=str, default="test-V-1.csv")

    args, _ = parser.parse_known_args()

    print("SKLearn Version: ", sklearn.__version__)
    print("Jobliv Version: ", joblib.__version__)

    print("[INFO] Reading data")
    print()
    train_df = pd.read_csv(os.path.join(args.train, args.train_file))
    test_df = pd.read_csv(os.path.join(args.test, args.test_file))

    features = list(train_df.columns)
    label = features.pop(-1)

    print("Building training and testing datasets")
    print()
    X_train = train_df[features]
    X_test = test_df[features]
    y_train = train_df[label]
    y_test = test_df[label]

    print('Column order: ')
    print(features)
    print()

    print("Label column is: ", label)
    print()

    print("Data shape: ")
    print()
    print("Shape of training data (85%):")
    print(X_train.shape)
    print(y_train.shape)
    print()
    print("Shape of testing data (15%):")
    print(X_test.shape)
    print(y_test.shape)
    print()

    print("Training RandomForest Model...")
    print()
    model=RandomForestClassifier(n_estimators=args.n_estimators, random_state=args.random_state,\
                                 verbose=2, n_jobs=1)

    model.fit(X_train,y_train)

    print()

    model_path = os.path.join(args.model_dir, "model.joblib")
    joblib.dump(model, model_path)

    print("Model saved at " + model_path)

    y_pred_test = model.predict(X_test)
    test_acc = accuracy_score(y_test, y_pred_test)
    test_rep = classification_report(y_test, y_pred_test)

    print()
    print("Metrics on test")
    print("Total rows are: ", X_test.shape[0])
    print('[TESTING] Model accuracy is: ', test_acc)
    print('[TESTING] Testing report: ')
    print(test_rep)
    


    



### AWS Sagemaker Entry point to Execute the Training script

In [ ]:
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import (
    Compute,
    SourceCode,
    StoppingCondition,
)



# 1. Retrieve the Scikit-learn ECR image URI using sess
sess = Session()
image_uri = image_uris.retrieve(
    framework="sklearn",
    region=sess.boto_region_name,
    version="1.2-1",
    py_version="py3",
    instance_type="ml.m5.large"
)

# 2. Configure Compute hardware & enable spot training
compute_config = Compute(
    instance_type="ml.m5.large",
    instance_count=1,
    enable_managed_spot_training=True,
)

# 3. Configure StoppingCondition timeouts
stopping_config = StoppingCondition(
    max_runtime_in_seconds=3600,     # Max training time: 1 hour
    max_wait_time_in_seconds=7200,    # Max total spot wait time: 2 hours
)


# 4. Configure Source Code (source_dir is required when entry_script is provided)
source_config = SourceCode(
    source_dir="src",                   # Current directory where script.py lives
    entry_script="script.py"
)

# 5. Instantiate ModelTrainer
sklearn_trainer = ModelTrainer(
    training_image=image_uri,
    role=role,
    source_code=source_config,
    compute=compute_config,
    stopping_condition=stopping_config,
    base_job_name="RF-custom-sklearn",
    hyperparameters={
        "n_estimators": "100",
        "random_state": "0",
    },
)

# To launch training in v3:
# from sagemaker.train.configs import InputData
# train_data = InputData(channel_name="train", data_source="s3://your-bucket/train.csv")
# sklearn_trainer.train(input_data_config=[train_data])

In [ ]:
from sagemaker.train.configs import Compute, StoppingCondition

print("Compute fields:", list(Compute.model_fields.keys()))
print("StoppingCondition fields:", list(StoppingCondition.model_fields.keys()))

In [ ]:
# launch training
from sagemaker.train.configs import InputData
inputs = [
    InputData(channel_name="train", data_source=trainpath),
    InputData(channel_name="test", data_source=testpath),
]
sklearn_trainer.train(input_data_config=inputs)

In [ ]:
import boto3

# Get the exact training job name from SageMaker v3 TrainingJob object
job_name = sklearn_trainer._latest_training_job.training_job_name
print(f"Fetching CloudWatch logs for Training Job: {job_name}\n")

# Initialize CloudWatch Logs client
logs_client = boto3.client("logs")
log_group = "/aws/sagemaker/TrainingJobs"

try:
    # Fetch log streams for this job
    streams_response = logs_client.describe_log_streams(
        logGroupName=log_group, logStreamNamePrefix=job_name
    )
    streams = streams_response.get("logStreams", [])

    if not streams:
        print("No log streams found for this job yet.")
    else:
        # Loop through streams and output log messages
        for stream in streams:
            events_response = logs_client.get_log_events(
                logGroupName=log_group,
                logStreamName=stream["logStreamName"],
                limit=100,
            )
            for event in events_response.get("events", []):
                print(event["message"].strip())

except Exception as e:
    print(f"Error fetching CloudWatch logs: {e}")

##  To get the model from S3

In [ ]:
sklearn_trainer._latest_training_job.wait(logs=False)

artifact = sm_boto3.describe_training_job(
    TrainingJobName=sklearn_trainer._latest_training_job.training_job_name
)["ModelArtifacts"]["S3ModelArtifacts"]

In [ ]:
artifact

## Deploy the model for endpoints

In [ ]:
import inspect
print(inspect.signature(ModelBuilder.__init__))

In [ ]:
import os
from time import gmtime, strftime
from sagemaker.core import image_uris
from sagemaker.serve.model_builder import ModelBuilder

#os.makedirs("/tmp/sagemaker", exist_ok=True)

model_name = f"Custom-sklearn-model-{strftime('%Y-%m-%d-%H-%M-%S', gmtime())}"

image_uri = image_uris.retrieve(
    framework="sklearn",
    region=sess.boto_region_name,
    version="1.2-1",
    py_version="py3",
    instance_type="ml.m5.large"
)

model_builder = ModelBuilder(
    s3_model_data_url=artifact,
    image_uri=image_uri,
    role_arn=role,
    sagemaker_session=sess,
    env_vars={
        "SAGEMAKER_PROGRAM": "script.py",
        "SAGEMAKER_SUBMIT_DIRECTORY": "src",
    }
)

# Crea il modello
model = model_builder.build(model_name=model_name)

In [ ]:
model

In [ ]:
## Endpoint deployment
endpoint_name = f"Custom-sklearn-model-{strftime('%Y-%m-%d-%H-%M-%S', gmtime())}"
print("EndpointName={}".format(endpoint_name))

predictor = model_builder.deploy(
    initial_instance_count=1,
    instance_type="ml.m4.xlarge",
    endpoint_name=endpoint_name
)